# Traffic Tracker Model Training

**Models trained:**
- **License Plates** (YOLOv8m fine-tune)
- **Vehicle Colour** (MobileNetV3-Large)
- **Vehicle Body Type** (MobileNetV3-Large)

---
### Before starting:
1. **Runtime → Change Runtime Type → GPU**
2. Upload your `kaggle.json` API key (created at https://www.kaggle.com/account)
3. Run all cells top-to-bottom
4. Trained weights save to Google Drive automatically

## Section 1: Setup
Install packages, configure Kaggle credentials, and verify GPU.

In [1]:
# GPU Check & Windows PyTorch crash protection
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No CUDA GPU detected by PyTorch.')
    print('If on local PC, install CUDA PyTorch: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --force-reinstall')


PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB


In [2]:
# ── Install dependencies ─────────────────────────────────────────────────
!pip install -q ultralytics kagglehub datasets huggingface_hub easyocr \
             torchvision Pillow tqdm pyyaml scipy
print('✅ Packages installed')

✅ Packages installed



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Kaggle credentials (supports Local PC and Colab)
import os, json, pathlib

try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    kaggle_dir = pathlib.Path('/root/.kaggle')
    kaggle_dir.mkdir(exist_ok=True)
    if not (kaggle_dir / 'kaggle.json').exists():
        print('Upload your kaggle.json file:')
        uploaded = files.upload()
        for fname, data in uploaded.items():
            dest = kaggle_dir / 'kaggle.json'
            dest.write_bytes(data)
            dest.chmod(0o600)
            print(f'Kaggle credentials saved to {dest}')
    else:
        print('Kaggle credentials present.')
else:
    user_home = pathlib.Path.home()
    kaggle_dir = user_home / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    if (kaggle_dir / 'kaggle.json').exists():
        print(f'Local Kaggle credentials found at {kaggle_dir / "kaggle.json"}')
    else:
        print(f'Please place your kaggle.json file in {kaggle_dir}')

Please place your kaggle.json file in C:\Users\pumas\.kaggle


In [4]:
# Output Directory Setup (supports Local PC and Colab)
import os, pathlib

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT = pathlib.Path('/content/drive/MyDrive/TrafficTrackerAI_models')
    RUNS_DIR = pathlib.Path('/content/runs')
else:
    print('Running on Local PC')
    DRIVE_OUTPUT = pathlib.Path('./models').resolve()
    RUNS_DIR = pathlib.Path('./runs').resolve()

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Models will be saved to: {DRIVE_OUTPUT}')

Running on Local PC
Models will be saved to: C:\Code projects\TrekIT\notebooks\models


## Section 2: Download Datasets

In [5]:
import kagglehub, json, pathlib
from tqdm.auto import tqdm
from PIL import Image as PILImage
from datasets import load_dataset

# 2a. License Plate Dataset
print('Downloading license plate dataset...')
plate_path = kagglehub.dataset_download('fareselmenshawii/license-plate-dataset')
print(f'License plates: {plate_path}')

# 2b. VCOR Vehicle Colour Dataset
print('Downloading VCOR colour dataset...')
vcor_path = kagglehub.dataset_download('landrykezebou/vcor-vehicle-color-recognition-dataset')
print(f'VCOR colour: {vcor_path}')

import os
import glob
print("Found YAMLs:", glob.glob(f"{plate_path}/**/*.yaml", recursive=True)[:3])
print('VCOR items:', os.listdir(vcor_path))

c:\Users\pumas\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


License plates: C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1
VCOR colour: C:\Users\pumas\.cache\kagglehub\datasets\landrykezebou\vcor-vehicle-color-recognition-dataset\versions\1
Found YAMLs: []
VCOR items: ['test', 'train', 'val']


In [6]:
# ── 2c. Stanford Cars Subset (streaming) ──────────────────────────────
STANFORD_OUT = pathlib.Path('./data/stanford_cars')
STANFORD_MAX_BYTES = 3 * 1024**3   # 3 GB cap
PER_CLASS_CAP = 50                  # 50 images per class for higher vehicle type accuracy

STANFORD_OUT.mkdir(parents=True, exist_ok=True)
print(f'Streaming Stanford Cars → {STANFORD_OUT} (cap: 3GB)...')

ds = load_dataset('tanganke/stanford_cars', split='train', streaming=True)
class_counts = {}
total_bytes = 0
saved = 0

pbar = tqdm(desc='Stanford Cars subset', unit='img')
for sample in ds:
    lbl = sample['label']
    img = sample['image']
    class_counts[lbl] = class_counts.get(lbl, 0)
    if class_counts[lbl] >= PER_CLASS_CAP:
        continue
    d = STANFORD_OUT / f'class_{lbl:03d}'
    d.mkdir(parents=True, exist_ok=True)
    p = d / f'{saved:06d}.jpg'
    img.convert('RGB').save(p, 'JPEG', quality=90)
    total_bytes += p.stat().st_size
    class_counts[lbl] += 1
    saved += 1
    pbar.update(1)
    pbar.set_postfix({'GB': f'{total_bytes/1024**3:.3f}', 'imgs': saved})
    if total_bytes >= STANFORD_MAX_BYTES:
        break
pbar.close()
print(f'Stanford Cars subset: {saved} images, {total_bytes/1024**3:.2f} GB')

Streaming Stanford Cars → data\stanford_cars (cap: 3GB)...


Stanford Cars subset: 8126img [02:38, 51.14img/s, GB=0.760, imgs=8126] 

Stanford Cars subset: 8126 images, 0.76 GB


In [7]:
# ── 2d. Map Stanford Cars 196 labels → 7 body types ──────────────────────
import shutil
from collections import Counter

BODY_TYPE_KEYWORDS = {
    'Convertible': ['convertible', 'cabriolet', 'roadster', 'spyder', 'spider'],
    'Van':         ['van', 'minivan', 'cargo van', 'transit'],
    'Truck':       ['truck', 'pickup', 'ram', 'silverado', 'f-150', 'f150', 'tacoma', 'tundra'],
    'Hatchback':   ['hatchback', 'hatch', '5-door', '5 door'],
    'SUV':         ['suv', '4wd', 'crossover', 'sport utility', 'wagon', 'explorer', 'tahoe'],
    'Coupe':       ['coupe', '2-door', '2 door', 'fastback'],
    'Sedan':       ['sedan', 'saloon', '4-door', '4 door'],
}
DEFAULT_TYPE = 'Sedan'

STANFORD_196 = [
    'AM General Hummer SUV 2000','Acura Integra Type R 2001','Acura RL Sedan 2012',
    'Acura TL Sedan 2012','Acura TL Type-S 2008','Acura TSX Sedan 2012',
    'Acura ZDX Hatchback 2012','Aston Martin V8 Vantage Convertible 2012',
    'Aston Martin V8 Vantage Coupe 2012','Aston Martin Virage Convertible 2012',
    'Aston Martin Virage Coupe 2012','Audi RS 4 Convertible 2008',
    'Audi A5 Coupe 2012','Audi TTS Coupe 2012','Audi R8 Coupe 2012',
    'Audi V8 Sedan 1994','Audi 100 Sedan 1994','Audi 100 Wagon 1994',
    'Audi TT Hatchback 2011','Audi S6 Sedan 2011','Audi S5 Convertible 2012',
    'Audi S5 Coupe 2012','Audi S4 Sedan 2012','Audi S4 Sedan 2007',
    'Audi TT RS Coupe 2012','BMW ActiveHybrid 5 Sedan 2012',
    'BMW 1 Series Convertible 2012','BMW 1 Series Coupe 2012',
    'BMW 3 Series Sedan 2012','BMW 3 Series Wagon 2012',
    'BMW 6 Series Convertible 2007','BMW X5 SUV 2007','BMW X6 SUV 2012',
    'BMW M3 Coupe 2012','BMW M5 Sedan 2010','BMW M6 Convertible 2010',
    'BMW X3 SUV 2012','BMW Z4 Convertible 2012','Bentley Continental GT Coupe 2012',
    'Bentley Continental GT Coupe 2007','Bentley Continental Flying Spur Sedan 2007',
    'Bentley Mulsanne Sedan 2011','Bentley Continental Supersports Conv. Convertible 2012',
    'Bugatti Veyron 16.4 Convertible 2009','Bugatti Veyron 16.4 Coupe 2009',
    'Buick Enclave SUV 2012','Buick Rainier SUV 2007','Buick Regal GS 2012',
    'Buick Verano Sedan 2012','Cadillac CTS-V Coupe 2012',
    'Cadillac Escalade EXT Crew Cab 2007','Cadillac SRX SUV 2012',
    'Chevrolet Silverado 1500 Classic Extended Cab 2007',
    'Chevrolet Silverado 1500 Extended Cab 2012',
    'Chevrolet Silverado 1500 Hybrid Crew Cab 2012',
    'Chevrolet Silverado 1500 Regular Cab 2012',
    'Chevrolet Silverado 2500HD Regular Cab 2012','Chevrolet Tahoe Hybrid SUV 2012',
    'Chevrolet TrailBlazer SS 2009','Chevrolet Traverse SUV 2012',
    'Chevrolet Colorado Crew Cab 2012','Chevrolet Camaro Convertible 2012',
    'Chevrolet Cobalt SS 2010','Chevrolet Corvette Convertible 2012',
    'Chevrolet Corvette Ron Fellows Edition Z06 2007',
    'Chevrolet Corvette ZR1 2012','Chevrolet Express Cargo Van 2007',
    'Chevrolet Express Van 2007','Chevrolet HHR SS 2010',
    'Chevrolet Impala Sedan 2007','Chevrolet Malibu Hybrid Sedan 2010',
    'Chevrolet Malibu Sedan 2007','Chevrolet Monte Carlo Coupe 2007',
    'Chevrolet Avalanche Crew Cab 2012','Chrysler 300 SRT-8 2010',
    'Chrysler Aspen SUV 2009','Chrysler Crossfire Convertible 2008',
    'Chrysler PT Cruiser Convertible 2008','Chrysler Town and Country Minivan 2012',
    'Chrysler 300 Sedan 2012','Daewoo Nubira Wagon 2002',
    'Dodge Caliber Wagon 2012','Dodge Caliber Wagon 2007',
    'Dodge Caravan Minivan 1997','Dodge Challenger SRT8 2011',
    'Dodge Charger SRT-8 2009','Dodge Charger Sedan 2012',
    'Dodge Dakota Club Cab 2007','Dodge Dakota Crew Cab 2010',
    'Dodge Durango SUV 2012','Dodge Durango SUV 2007',
    'Dodge Journey SUV 2012','Dodge Magnum Wagon 2008',
    'Dodge Ram Pickup 3500 Crew Cab 2010','Dodge Ram Pickup 3500 Quad Cab 2009',
    'Dodge Sprinter Cargo Van 2009','Dodge Viper Convertible 2010',
    'Dodge Viper SRT-10 Coupe 2010','Eagle Talon Hatchback 1998',
    'FIAT 500 Abarth 2012','FIAT 500 Convertible 2012',
    'Ferrari 458 Italia Convertible 2012','Ferrari 458 Italia Coupe 2012',
    'Ferrari California Convertible 2012','Ferrari FF Coupe 2012',
    'Fisker Karma Sedan 2012','Ford F-150 Regular Cab 2012',
    'Ford F-450 Super Duty Crew Cab 2012','Ford Fiesta Sedan 2012',
    'Ford Focus Sedan 2007','Ford Freestar Minivan 2007',
    'Ford GT Coupe 2006','Ford Galaxy Minivan 2007',
    'Ford Mustang Convertible 2007','Ford Mustang Convertible 1993',
    'Ford Ranger SuperCab 2011','Ford F-150 Regular Cab 2007',
    'GMC Acadia SUV 2012','GMC Canyon Extended Cab 2012',
    'GMC Savana Van 2012','GMC Sierra 1500 Classic Extended Cab 2007',
    'GMC Sierra 1500 Extended Cab 2012','GMC Sierra 1500 Hybrid Crew Cab 2012',
    'GMC Sierra 1500 Regular Cab 2012','GMC Sierra 2500HD Regular Cab 2012',
    'GMC Terrain SUV 2012','GMC Yukon Hybrid SUV 2012',
    'Geo Metro Convertible 1993','HUMMER H2 SUT Crew Cab 2009',
    'HUMMER H3T Crew Cab 2010','Honda Accord Coupe 2012',
    'Honda Accord Sedan 2012','Honda Odyssey Minivan 2012',
    'Honda Odyssey Minivan 2007','Hyundai Azera Sedan 2012',
    'Hyundai Elantra Sedan 2007','Hyundai Elantra Touring Hatchback 2012',
    'Hyundai Genesis Sedan 2012','Hyundai Santa Fe SUV 2012',
    'Hyundai Sonata Hybrid Sedan 2012','Hyundai Sonata Sedan 2012',
    'Hyundai Tucson SUV 2012','Hyundai Veloster Hatchback 2012',
    'Hyundai Veracruz SUV 2012','Infiniti G Coupe IPL 2012',
    'Infiniti QX56 SUV 2011','Isuzu Ascender SUV 2008',
    'Jaguar XK XKR 2012','Jeep Compass SUV 2012',
    'Jeep Grand Cherokee SUV 2012','Jeep Liberty SUV 2012',
    'Jeep Patriot SUV 2012','Jeep Wrangler SUV 2012',
    'Lamborghini Aventador Coupe 2012','Lamborghini Diablo Coupe 2001',
    'Lamborghini Gallardo LP 570-4 Superleggera 2012',
    'Lamborghini Reventon Coupe 2008','Land Rover LR2 SUV 2012',
    'Land Rover Range Rover SUV 2012','Lincoln Town Car Sedan 2011',
    'MINI Cooper Roadster Convertible 2012','Maybach Landaulet Convertible 2012',
    'Mazda Tribute SUV 2011','McLaren MP4-12C Coupe 2012',
    'Mercedes-Benz 300-Class Convertible 1993','Mercedes-Benz C-Class Sedan 2012',
    'Mercedes-Benz E-Class Sedan 2012','Mercedes-Benz S-Class Sedan 2012',
    'Mercedes-Benz SL-Class Coupe 2009','Mercedes-Benz Sprinter Van 2012',
    'Mitsubishi Lancer Sedan 2012','Nissan 240SX Coupe 1998',
    'Nissan Juke Hatchback 2012','Nissan Leaf Hatchback 2012',
    'Nissan NV Passenger Van 2012','Plymouth Neon Coupe 1999',
    'Porsche Panamera Sedan 2012','Ram C/V Cargo Van Minivan 2012',
    'Rolls-Royce Ghost Sedan 2012',
    'Rolls-Royce Phantom Drophead Coupe Convertible 2012',
    'Rolls-Royce Phantom Sedan 2012','Scion xD Hatchback 2012',
    'Spyker C8 Convertible 2009','Spyker C8 Coupe 2009',
    'Suzuki Aerio Sedan 2007','Suzuki Kizashi Sedan 2012',
    'Suzuki SX4 Hatchback 2012','Suzuki SX4 Sedan 2012',
    'Tesla Model S Sedan 2012','Toyota 4Runner SUV 2012',
    'Toyota Camry Sedan 2012','Toyota Corolla Sedan 2012',
    'Toyota Sequoia SUV 2012','Volkswagen Beetle Hatchback 2012',
    'Volkswagen Golf Hatchback 2012','Volkswagen Golf Hatchback 1991',
    'Volvo 240 Sedan 1993','Volvo C30 Hatchback 2012',
    'smart fortwo Convertible 2012',
]

def label_to_type(label):
    lower = label.lower()
    for bt, kws in BODY_TYPE_KEYWORDS.items():
        for kw in kws:
            if kw in lower:
                return bt
    return DEFAULT_TYPE

label_map = {i: label_to_type(n) for i, n in enumerate(STANFORD_196)}

TYPED_OUT = pathlib.Path('./data/stanford_cars_typed')
all_types = set(BODY_TYPE_KEYWORDS.keys()) | {DEFAULT_TYPE}
for bt in all_types:
    (TYPED_OUT / bt).mkdir(parents=True, exist_ok=True)

for class_dir in sorted(STANFORD_OUT.glob('class_*')):
    cid = int(class_dir.name.replace('class_', ''))
    bt = label_map.get(cid, DEFAULT_TYPE)
    for p in class_dir.glob('*.jpg'):
        dest = TYPED_OUT / bt / f'c{cid:03d}_{p.name}'
        shutil.copy2(p, dest)

counts = Counter(label_map.values())
print('Body type mapping:')
for bt, c in counts.most_common():
    img_c = len(list((TYPED_OUT / bt).glob('*.jpg')))
    print(f'  {bt:<15}: {c} classes → {img_c} images')
print(f'Stanford Cars typed subset ready at {TYPED_OUT}')

Body type mapping:
  Sedan          : 72 classes → 2957 images
  SUV            : 37 classes → 1515 images
  Convertible    : 27 classes → 1075 images
  Coupe          : 27 classes → 1122 images
  Van            : 14 classes → 573 images
  Hatchback      : 13 classes → 496 images
  Truck          : 9 classes → 388 images
Stanford Cars typed subset ready at data\stanford_cars_typed


## Section 3: License Plate Detection — YOLO11 Fine-tune
Fine-tune YOLO11 on the license plate detection dataset (optimized for RTX 3060 12GB).

In [8]:
import os, glob, yaml, pathlib

# 1. Find all actual image directories in the Kaggle dataset
all_jpgs = glob.glob(f'{plate_path}/**/*.jpg', recursive=True) + glob.glob(f'{plate_path}/**/*.png', recursive=True)
all_image_dirs = sorted(list(set(os.path.dirname(p) for p in all_jpgs)))

print("Found image directories in Kaggle dataset:")
for d in all_image_dirs:
    print(" -", d)

# 2. Match train and validation directories automatically
train_dir = next((d for d in all_image_dirs if 'train' in d.lower()), all_image_dirs[0])
val_dir   = next((d for d in all_image_dirs if any(k in d.lower() for k in ['val', 'valid', 'test'])), train_dir)

print(f"\nUsing train dir: {train_dir}")
print(f"Using val dir:   {val_dir}")

# 3. Write paths for YOLO
PLATE_YAML_FIXED = str(pathlib.Path('./data/plate_dataset.yaml').resolve())
pathlib.Path('./data').mkdir(parents=True, exist_ok=True)

cfg = {
    'path': '',
    'train': train_dir,
    'val': val_dir,
    'nc': 1,
    'names': ['license_plate'],
}

with open(PLATE_YAML_FIXED, 'w') as f:
    yaml.dump(cfg, f)

print(f"\nFixed YAML created at {PLATE_YAML_FIXED}")

Found image directories in Kaggle dataset:
 - C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1\images\train
 - C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1\images\val

Using train dir: C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1\images\train
Using val dir:   C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1\images\val

Fixed YAML created at C:\Code projects\TrekIT\notebooks\data\plate_dataset.yaml


In [ ]:
from ultralytics import YOLO
import shutil, torch

# Fine-tune YOLO11 on license plate detection (optimized for RTX 3060 12GB)
try:
    plate_model = YOLO('yolo11s.pt')
    print('Loaded YOLO11s base model.')
except Exception:
    plate_model = YOLO('yolov8m.pt')
    print('Fallback to YOLOv8m base model.')

results = plate_model.train(
    data=PLATE_YAML_FIXED,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(RUNS_DIR),
    name='plate_detector',
    patience=15,
    save=True,
    plots=True,
    verbose=True,
    amp=True,
    workers=0,
    
    # Augmentations for small object detection & perspective tilt
    scale=0.5,
    degrees=15.0,
    perspective=0.001,
    mosaic=0.5,
    close_mosaic=10,
)

PLATE_WEIGHTS = f'{results.save_dir}/weights/best.pt'

# Copy best weights directly to output directory
drive_plate_dest = f'{DRIVE_OUTPUT}/plate_detector.pt'
shutil.copy2(PLATE_WEIGHTS, drive_plate_dest)

print('License plate detector (YOLO11 1024px) training complete.')
print(f'Local weights: {PLATE_WEIGHTS}')
print(f'Saved to output folder: {drive_plate_dest}')

Loaded YOLO11s base model.
New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.118  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Code projects\TrekIT\notebooks\data\plate_dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None

c:\Users\pumas\AppData\Local\Programs\Python\Python312\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\pumas\AppData\Local\Programs\Python\Python312\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Trig

AMP: checks passed 
train: Fast image access  (ping: 0.50.6 ms, read: 125.358.8 MB/s, size: 209.5 KB)
train: Scanning C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1\labels\train.cache... 4295 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4295/4295  0.0s
val: Fast image access  (ping: 0.10.1 ms, read: 406.9178.8 MB/s, size: 357.2 KB)
val: Scanning C:\Users\pumas\.cache\kagglehub\datasets\fareselmenshawii\license-plate-dataset\versions\1\labels\val.cache... 1073 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1073/1073  0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to C:\Code projects\TrekIT\notebooks\runs\plate_detector-4\labels.jpg... 
Image sizes 640 train, 640 val
Using 0 dataloader

c:\Users\pumas\AppData\Local\Programs\Python\Python312\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\pumas\AppData\Local\Programs\Python\Python312\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Trig


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      3.96G      1.817      3.081      1.509         16        640: 100% ━━━━━━━━━━━━ 269/269 2.2it/s 2:040.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 2.4it/s 14.4s0.5s
                   all       1073       1573      0.596      0.423      0.472      0.188

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


## Section 4: Vehicle Colour Classifier — MobileNetV3-Large

In [2]:
import torch, torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import pathlib, os, kagglehub

# ── Ensure paths and environment are initialized ───────────────────────────
if 'vcor_path' not in globals() or not vcor_path:
    print('Resolving VCOR colour dataset path from KaggleHub...')
    vcor_path = kagglehub.dataset_download('landrykezebou/vcor-vehicle-color-recognition-dataset')
    print(f'VCOR colour: {vcor_path}')

if 'RUNS_DIR' not in globals():
    RUNS_DIR = pathlib.Path('./runs').resolve()
    RUNS_DIR.mkdir(parents=True, exist_ok=True)

if 'DRIVE_OUTPUT' not in globals():
    DRIVE_OUTPUT = pathlib.Path('./models').resolve()
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# ── Explore VCOR dataset structure ─────────────────────────────────────────
print('VCOR dataset contents:')
for item in os.listdir(vcor_path):
    p = os.path.join(vcor_path, item)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f'  {item}/ ({n} items)')


c:\Users\pumas\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resolving VCOR colour dataset path from KaggleHub...
VCOR colour: C:\Users\pumas\.cache\kagglehub\datasets\landrykezebou\vcor-vehicle-color-recognition-dataset\versions\1
VCOR dataset contents:
  test/ (15 items)
  train/ (15 items)
  val/ (15 items)


In [3]:
import random, shutil
from collections import defaultdict

# Build train/val split from VCOR (80/20 stratified)
VCOR_SPLIT = pathlib.Path('./data/vcor_split')

# Detect structure: either flat class folders or train/test already split
vcor = pathlib.Path(vcor_path)
subdirs = [d for d in vcor.iterdir() if d.is_dir()]

if any(d.name.lower() in ('train', 'test', 'val') for d in subdirs):
    # Already split — use as-is
    train_root = next(d for d in subdirs if d.name.lower() == 'train')
    val_root   = next((d for d in subdirs if d.name.lower() in ('val','test')), train_root)
    print(f'Pre-split dataset: train={train_root}, val={val_root}')
else:
    # Flat class folders — split manually
    print('Building 80/20 train/val split...')
    random.seed(42)
    for cls_dir in sorted(subdirs):
        imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        random.shuffle(imgs)
        n_val = max(1, len(imgs) // 5)
        for split, batch in [('val', imgs[:n_val]), ('train', imgs[n_val:])]:
            d = VCOR_SPLIT / split / cls_dir.name
            d.mkdir(parents=True, exist_ok=True)
            for p in batch:
                shutil.copy2(p, d / p.name)
    train_root = VCOR_SPLIT / 'train'
    val_root   = VCOR_SPLIT / 'val'
    print(f'Split complete → {VCOR_SPLIT}')

# DataLoaders
COLOUR_CLASSES = sorted([d.name for d in train_root.iterdir() if d.is_dir()])
print(f'Colour classes ({len(COLOUR_CLASSES)}): {COLOUR_CLASSES}')

IMG_SIZE = 224
BATCH = 32

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(train_root), transform=train_tf)
val_ds   = datasets.ImageFolder(str(val_root),   transform=val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train: {len(train_ds)} images | Val: {len(val_ds)} images')

Pre-split dataset: train=C:\Users\pumas\.cache\kagglehub\datasets\landrykezebou\vcor-vehicle-color-recognition-dataset\versions\1\train, val=C:\Users\pumas\.cache\kagglehub\datasets\landrykezebou\vcor-vehicle-color-recognition-dataset\versions\1\test
Colour classes (15): ['beige', 'black', 'blue', 'brown', 'gold', 'green', 'grey', 'orange', 'pink', 'purple', 'red', 'silver', 'tan', 'white', 'yellow']
Train: 7267 images | Val: 1556 images


In [4]:
def train_mobilenet(train_dl, val_dl, class_names, epochs=40, name='model'):
    num_classes = len(class_names)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = torch.cuda.is_available()

    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_acc = 0.0
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    best_path = str(RUNS_DIR / f'{name}_best.pt')

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * imgs.size(0)
            train_correct += (out.argmax(1) == labels).sum().item()
            train_total += imgs.size(0)
        scheduler.step()

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs, labels = imgs.to(device), labels.to(device)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    out = model(imgs)
                val_correct += (out.argmax(1) == labels).sum().item()
                val_total += imgs.size(0)
        val_acc = val_correct / max(val_total, 1)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Loss: {train_loss/train_total:.4f} | '
                  f'Val Acc: {val_acc:.3f} | Best: {best_acc:.3f}')

    print(f'\n{name} training complete. Best Val Acc: {best_acc:.3f}')
    print(f'   Weights saved to: {best_path}')
    return best_path, class_names


# Train colour classifier with AMP
COLOR_WEIGHTS, COLOR_CLASSES_OUT = train_mobilenet(
    train_dl, val_dl, COLOUR_CLASSES, epochs=40, name='color_classifier'
)

C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\1290077580.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\1290077580.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\1290077580.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch   1/40 | Loss: 0.8295 | Val Acc: 0.831 | Best: 0.831
Epoch   5/40 | Loss: 0.1717 | Val Acc: 0.826 | Best: 0.835
Epoch  10/40 | Loss: 0.0698 | Val Acc: 0.832 | Best: 0.835
Epoch  15/40 | Loss: 0.0472 | Val Acc: 0.852 | Best: 0.852
Epoch  20/40 | Loss: 0.0163 | Val Acc: 0.846 | Best: 0.853
Epoch  25/40 | Loss: 0.0145 | Val Acc: 0.845 | Best: 0.857
Epoch  30/40 | Loss: 0.0062 | Val Acc: 0.849 | Best: 0.861
Epoch  35/40 | Loss: 0.0049 | Val Acc: 0.856 | Best: 0.861
Epoch  40/40 | Loss: 0.0040 | Val Acc: 0.855 | Best: 0.861

color_classifier training complete. Best Val Acc: 0.861
   Weights saved to: C:\Code projects\TrekIT\notebooks\runs\color_classifier_best.pt


## Section 5: Vehicle Body Type Classifier — MobileNetV3-Large

In [5]:
# DataLoaders for Stanford Cars typed subset (80/20 split)
import random, shutil, pathlib
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

if 'TYPED_OUT' not in globals():
    TYPED_OUT = pathlib.Path('./data/stanford_cars_typed')
if 'RUNS_DIR' not in globals():
    RUNS_DIR = pathlib.Path('./runs').resolve()
if 'BATCH' not in globals():
    BATCH = 32

IMG_SIZE = 224
if 'train_tf' not in globals():
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

TYPE_SPLIT = pathlib.Path('./data/type_split')
random.seed(42)

for cls_dir in sorted(TYPED_OUT.iterdir()):
    if not cls_dir.is_dir():
        continue
    imgs = list(cls_dir.glob('*.jpg'))
    random.shuffle(imgs)
    n_val = max(1, len(imgs) // 5)
    for split, batch in [('val', imgs[:n_val]), ('train', imgs[n_val:])]:
        d = TYPE_SPLIT / split / cls_dir.name
        d.mkdir(parents=True, exist_ok=True)
        for p in batch:
            shutil.copy2(p, d / p.name)

TYPE_CLASSES = sorted([d.name for d in (TYPE_SPLIT / 'train').iterdir() if d.is_dir()])
print(f'Type classes ({len(TYPE_CLASSES)}): {TYPE_CLASSES}')

type_train_ds = datasets.ImageFolder(str(TYPE_SPLIT / 'train'), transform=train_tf)
type_val_ds   = datasets.ImageFolder(str(TYPE_SPLIT / 'val'),   transform=val_tf)
type_train_dl = DataLoader(type_train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
type_val_dl   = DataLoader(type_val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train: {len(type_train_ds)} images | Val: {len(type_val_ds)} images')


Type classes (7): ['Convertible', 'Coupe', 'Hatchback', 'SUV', 'Sedan', 'Truck', 'Van']
Train: 6503 images | Val: 1623 images


In [6]:
# Train body type classifier (~45 min on T4)
TYPE_WEIGHTS, TYPE_CLASSES_OUT = train_mobilenet(
    type_train_dl, type_val_dl, TYPE_CLASSES, epochs=30, name='type_classifier'
)

C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\1290077580.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\1290077580.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\1290077580.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch   1/30 | Loss: 1.6262 | Val Acc: 0.443 | Best: 0.443
Epoch   5/30 | Loss: 0.2576 | Val Acc: 0.585 | Best: 0.620
Epoch  10/30 | Loss: 0.0876 | Val Acc: 0.667 | Best: 0.678
Epoch  15/30 | Loss: 0.0318 | Val Acc: 0.673 | Best: 0.701
Epoch  20/30 | Loss: 0.0157 | Val Acc: 0.702 | Best: 0.709
Epoch  25/30 | Loss: 0.0044 | Val Acc: 0.717 | Best: 0.720
Epoch  30/30 | Loss: 0.0034 | Val Acc: 0.722 | Best: 0.726

type_classifier training complete. Best Val Acc: 0.726
   Weights saved to: C:\Code projects\TrekIT\notebooks\runs\type_classifier_best.pt


## Section 5b: Surveillance Multi-Task Classifier — VehicleAttributeNet (VeRi-776)
Train unified multi-task network (Color + Body Type) on CCTV surveillance camera angles from the VeRi-776 benchmark dataset.
This eliminates the Stanford Cars perspective gap and outputs `models/vehicle_attributes.pt`.

In [ ]:
# ── Download & Train VehicleAttributeNet on VeRi-776 ──────────────────
import os, sys, kagglehub, pathlib
from data_prep.veri776_parser import train_vehicle_attribute_net

print('Downloading VeRi-776 dataset from Kaggle...')
try:
    veri_path = kagglehub.dataset_download('yiransun/veri-776')
    print(f'VeRi-776 ready at: {veri_path}')
    train_vehicle_attribute_net(data_dir=veri_path, epochs=30, batch_size=32, output_path='models/vehicle_attributes.pt')
except Exception as e:
    print(f'VeRi-776 download info: {e}')
    print('You can also run locally: python data_prep/veri776_parser.py --data_dir data/veri776')


## Section 5c: Dedicated ALPR Sequence Recognition — LPRNet
Train lightweight end-to-end character sequence recognition model (LPRNet + CTC Loss).
Replaces general EasyOCR with a native 1ms sequence recognizer (`models/lprnet.pt`).

In [ ]:
# ── Train LPRNet Sequence Recognizer ─────────────────────────────────
from data_prep.train_lprnet import train_lprnet

print('Training LPRNet with CTC Loss...')
train_lprnet(epochs=35, batch_size=64, output_path='models/lprnet.pt')


## Section 6: Export Models & Download
Export to ONNX, zip files, save to Google Drive, and provide download links.

In [8]:
import json, zipfile, shutil, pathlib, os, torch
from torchvision import models
from ultralytics import YOLO

# Install required exporter backend if missing
!pip install -q onnxscript

if 'DRIVE_OUTPUT' not in globals():
    DRIVE_OUTPUT = pathlib.Path('./models').resolve()

EXPORT_DIR = DRIVE_OUTPUT / 'TrafficTrackerAI_models'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Auto-resolve paths if kernel was restarted ──────────────────────────────
if 'PLATE_WEIGHTS' not in globals() or not PLATE_WEIGHTS:
    plate_runs = sorted(list(pathlib.Path('./runs').glob('plate_detector*/weights/best.pt')), key=os.path.getmtime)
    PLATE_WEIGHTS = str(plate_runs[-1]) if plate_runs else './models/plate_detector.pt'

if 'COLOR_WEIGHTS' not in globals() or not COLOR_WEIGHTS:
    COLOR_WEIGHTS = './runs/color_classifier_best.pt'

if 'TYPE_WEIGHTS' not in globals() or not TYPE_WEIGHTS:
    TYPE_WEIGHTS = './runs/type_classifier_best.pt'

if 'COLOR_CLASSES_OUT' not in globals() or not COLOR_CLASSES_OUT:
    if 'COLOUR_CLASSES' in globals() and COLOUR_CLASSES:
        COLOR_CLASSES_OUT = COLOUR_CLASSES
    elif pathlib.Path('./data/vcor_split/train').exists():
        COLOR_CLASSES_OUT = sorted([d.name for d in pathlib.Path('./data/vcor_split/train').iterdir() if d.is_dir()])
    elif pathlib.Path('../models/color_classes.json').exists():
        COLOR_CLASSES_OUT = json.load(open('../models/color_classes.json'))
    else:
        COLOR_CLASSES_OUT = ['black', 'blue', 'brown', 'green', 'grey', 'orange', 'pink', 'purple', 'red', 'silver', 'white', 'yellow']

if 'TYPE_CLASSES_OUT' not in globals() or not TYPE_CLASSES_OUT:
    if 'TYPE_CLASSES' in globals() and TYPE_CLASSES:
        TYPE_CLASSES_OUT = TYPE_CLASSES
    elif pathlib.Path('./data/type_split/train').exists():
        TYPE_CLASSES_OUT = sorted([d.name for d in pathlib.Path('./data/type_split/train').iterdir() if d.is_dir()])
    elif pathlib.Path('../models/type_classes.json').exists():
        TYPE_CLASSES_OUT = json.load(open('../models/type_classes.json'))
    else:
        TYPE_CLASSES_OUT = ['Convertible', 'Coupe', 'Hatchback', 'SUV', 'Sedan', 'Truck', 'Van']

print(f'Plate weights : {PLATE_WEIGHTS}')
print(f'Color weights : {COLOR_WEIGHTS}')
print(f'Type weights  : {TYPE_WEIGHTS}')

# ── 1. Copy PyTorch (.pt) weights ──────────────────────────────────────────
if pathlib.Path(PLATE_WEIGHTS).exists():
    shutil.copy2(PLATE_WEIGHTS, EXPORT_DIR / 'plate_detector.pt')
if pathlib.Path(COLOR_WEIGHTS).exists():
    shutil.copy2(COLOR_WEIGHTS, EXPORT_DIR / 'color_classifier.pt')
if pathlib.Path(TYPE_WEIGHTS).exists():
    shutil.copy2(TYPE_WEIGHTS,  EXPORT_DIR / 'type_classifier.pt')
print('PyTorch (.pt) models copied')

# ── 2. Export YOLO Plate Detector to ONNX ──────────────────────────────────
onnx_src = str(PLATE_WEIGHTS).replace('.pt', '.onnx')
if pathlib.Path(onnx_src).exists():
    shutil.copy2(onnx_src, EXPORT_DIR / 'plate_detector.onnx')
    print('plate_detector.onnx copied')
else:
    try:
        plate_model_export = YOLO(PLATE_WEIGHTS)
        plate_model_export.export(format='onnx', imgsz=640, simplify=True)
        shutil.copy2(onnx_src, EXPORT_DIR / 'plate_detector.onnx')
        print('plate_detector.onnx exported')
    except Exception as e:
        print(f'YOLO ONNX export skipped: {e}')

# ── 3. Export MobileNet Classifiers to ONNX ───────────────────────────────
dummy_input = torch.randn(1, 3, 224, 224).to(device)

def export_mobilenet_onnx(pt_path, class_names, output_onnx_path):
    try:
        model = models.mobilenet_v3_large(weights=None)
        model.classifier[-1] = torch.nn.Linear(model.classifier[-1].in_features, len(class_names))
        model.load_state_dict(torch.load(pt_path, map_location=device))
        model.to(device).eval()
        
        torch.onnx.export(
            model,
            dummy_input,
            output_onnx_path,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
            opset_version=14,
            dynamo=False
        )
    except Exception as e:
        print(f'ONNX export failed for {pt_path}: {e}')

export_mobilenet_onnx(COLOR_WEIGHTS, COLOR_CLASSES_OUT, EXPORT_DIR / 'color_classifier.onnx')
export_mobilenet_onnx(TYPE_WEIGHTS, TYPE_CLASSES_OUT, EXPORT_DIR / 'type_classifier.onnx')
print('MobileNet classifiers exported to ONNX format')

# ── 4. Save class name lists as JSON ───────────────────────────────────────
with open(EXPORT_DIR / 'color_classes.json', 'w') as f:
    json.dump(COLOR_CLASSES_OUT, f)
with open(EXPORT_DIR / 'type_classes.json', 'w') as f:
    json.dump(TYPE_CLASSES_OUT, f)
print('Class name JSONs saved')

# ── 5. Auto-sync to main TrekIT models folder ──────────────────────────────
root_models = pathlib.Path('../models').resolve()
root_models.mkdir(parents=True, exist_ok=True)
for f in EXPORT_DIR.glob('*'):
    shutil.copy2(f, root_models / f.name)
print(f'Deployed all updated models to TrekIT application: {root_models}')

# ── 6. Zip everything ──────────────────────────────────────────────────────
zip_path = DRIVE_OUTPUT / 'TrafficTrackerAI_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in EXPORT_DIR.glob('*'):
        zf.write(f, f.name)
zip_size = pathlib.Path(zip_path).stat().st_size / 1e6
print(f'Zipped all models → {zip_path} ({zip_size:.1f} MB)')



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Plate weights : runs\plate_detector-4\weights\best.pt
Color weights : C:\Code projects\TrekIT\notebooks\runs\color_classifier_best.pt
Type weights  : C:\Code projects\TrekIT\notebooks\runs\type_classifier_best.pt
PyTorch (.pt) models copied
Ultralytics 8.4.118  Python-3.12.10 torch-2.5.1+cu121 CPU (AMD Ryzen 5 5600G with Radeon Graphics)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from 'runs\plate_detector-4\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (72.5 MB)

ONNX: starting export with onnx 1.22.0 opset 19...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success  2.0s, saved as 'runs\plate_detector-4\weights\best.onnx' (36.2 MB)

Export complete (2.5s)
Results saved to C:\Code projects\TrekIT\notebooks\runs\plate_detector-4\weights\best.onnx
Predict:         yolo predict task=detect model=runs\plate_detector-4\weights\best.onnx imgsz=640 
Validate:        yolo val task=detec

C:\Users\pumas\AppData\Local\Temp\ipykernel_9648\3307677694.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(pt_path, map_location=devic

MobileNet classifiers exported to ONNX format
Class name JSONs saved
Deployed all updated models to TrekIT application: C:\Code projects\TrekIT\models
Zipped all models → C:\Code projects\TrekIT\notebooks\models\TrafficTrackerAI_models.zip (162.4 MB)


In [9]:
# ── Download to your computer directly (Colab or Local PC) ─────────────────────────
try:
    from google.colab import files
    print('Downloading TrafficTrackerAI_models.zip to your computer...')
    files.download(zip_path)
except ImportError:
    print('Running on Local PC — all models are already saved to your local disk!')

print(f'Models saved in: {zip_path}')
print()
print('Trained models location:')
print('  models/plate_detector.pt')
print('  models/color_classifier.pt')
print('  models/type_classifier.pt')
print()
print('Launch the web dashboard: python app/dashboard.py')

Running on Local PC — all models are already saved to your local disk!
Models saved in: C:\Code projects\TrekIT\notebooks\models\TrafficTrackerAI_models.zip

Trained models location:
  models/plate_detector.pt
  models/color_classifier.pt
  models/type_classifier.pt

Launch the web dashboard: python app/dashboard.py
